In [55]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests

In [56]:
from langchain.agents import create_agent

In [57]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERAPI_API_KEY = os.getenv("WEATHERAPI_API_KEY")

In [58]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
result = search_tool.invoke("Give me the latest news on AI")

[{'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth', 'url': 'https://www.artificialintelligence-news.com', 'content': 'AI in Action\n\nJuly 21, 2026\n\n### AWS GraphRAG deployment cuts drug research cycles by 87%\n\nHealthcare & Wellness AI\n\nJuly 9, 2026\n\n#### Industries\n\n### Zuckerberg details Meta’s personal AI superintelligence strategy\n\nAI and Us\n\nJuly 30, 2026\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nEnvironment & Sustainability\n\nMay 22, 2026\n\n### IBM Research unveils breakthrough analog AI chip for efficient deep learning [...] Inside AI\n

In [60]:
# ==========================================
# CUSTOM TOOL
# ==========================================

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherapi.com/v1//current.json?"
        f"key={WEATHERAPI_API_KEY}&q={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temp_c']}°C\n"
        f"Weather: {data['current']['condition']['text']}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [ ]:
# ==========================================
# LLM
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=GEMINI_API_KEY
)

In [ ]:
response = llm.invoke("Tell me a joke about AI")

content=[{'type': 'text', 'text': "**How many AIs does it take to change a lightbulb?**\n\nNone. It just confidently assures you the lightbulb *has* been changed, cites three fake academic sources to prove it, and asks if you'd like a poem about lamps.", 'extras': {'signature': 'Es4aCssaARFNMg9e3476xYPw3fIh1xsTBEeKgfmxyFBA2Ft2uNjoQezJGNViEGVrtIcIGjNBz9aO7IOzxrPCjxl6HDZloLw2ijZ6YEZSNb2Kr08t95cmCMWPaSWXBRDQp3t9w50v4ILtb4TtJZsV6/voyHM6N8NA9eNR21BfiSl28ezKi+7hh7b94ApQG7KAyixX5ZaeQu6+rn37Shcxs9Xagd9ylcmhUS19qBpe5y931/CrrfySVRTlzqQBlYbnDWJYX0NzIJ8sJQGUkSF8yXxCAz383aV11BUJtGjcH7G9gKDh1z3MJIoB80uX07Q4MmeLFNPtwMWN/0tqHmk/AHG/w40WoiKhIUdu5jzs4NNstvVg72GCv1LFtM+JX84fJw3tA9sLPwfhwM3B9wtBDut3ctVwbAjAMvi0Qlc/qyxYj0nrCCwF8pm8G8NGsGMh8H0x7Y1+JXOJo/nN1Ayh6suV4JImufDTjYZvg3wE8O/nzoX9LxFF7/ndyuTv8U28vPUpTTmGHC0xi1QYaLYkWawpihZ2JG+oT4+VrJicaEv40z5/dXQSLtK1ivHdJ9xt5UFwanakP1J5tE2adJVPU1aj4WNtziZGK7XnpZzUC9XIVS5MW89n6W0vfPL00z2pVNmq9UYZ3OmCB9r49oYomhlsnplmTQR5cUx0nKyWbWaMaqe428NldUam6UjuoVzgMbDYcmNVP+dc91hp

In [63]:
prompt=(
        "You are a helpful research assistant. "
        "Use the available tools when you need current or external information. "
        "For questions requiring multiple pieces of information, "
        "complete the necessary tool calls before giving the final answer."
)

In [64]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [65]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
)

In [66]:
# ==========================================
# RUN
# ==========================================

response = agent.invoke({
        "messages": [{
                "role": "user",
                "content": "What is the capital of India and what is the current weather there?"
            }]
})

In [67]:
print(response["messages"][-1].content[0]["text"])

The capital of India is **New Delhi**.

**Current Weather in New Delhi:**
* **Temperature:** 35.2°C
* **Conditions:** Partly Cloudy
* **Humidity:** 49%
